# HJM kernel covariance — derivation & walk-through

The `hjm_kernel` estimator replaces a noisy ~$n^2/2$-entry sample covariance with a smooth, PSD covariance
generated by a $k$-factor **Gaussian HJM** (Hull–White / LGM) model, fitting only $2k+\binom{k}{2}$
parameters. We (1) derive the covariance form, (2) reproduce
[`src/estimators/hjm_kernel.py`](../src/estimators/hjm_kernel.py) step by step, and (3) study the
dependence on the number of factors $k$.

## 1. Gaussian HJM setup

The forward rate follows $df(t,T)=\mu(t,T)\,dt+\sigma(t,T)\,dW_t$. In **Gaussian HJM** $\sigma$ is
deterministic, so $f$ is Gaussian. The drift $\mu$ is pinned by no-arbitrage but contributes only
$O(dt^2)$ to a covariance, so we keep the diffusion only.

## 2. Separability forces exponential loadings

A finite-dimensional Markov state requires $\sigma$ **separable**, $\sigma(t,T)=\sigma(t)\,G(t,T)$ with
$G(t,t)=1$. **Time-homogeneity** makes $G$ depend only on $\tau=T-t$, and composition gives the
multiplicative Cauchy equation $G(a+b)=G(a)G(b)$, so $G(\tau)=e^{-\lambda\tau}$. Hence the
**Hull–White / LGM** volatility
$$\sigma(t,T)=\sigma(t)\,e^{-\lambda(T-t)},$$
with $\lambda$ the decay speed and $\sigma(t)$ the shock scale.

## 3. Multi-factor superposition

One exponential is rank-1 (§6). Superpose $k$ correlated factors:
$$df(t,T)=\mu\,dt+\sum_{i=1}^{k}\sigma_i\,e^{-\lambda_i(T-t)}\,dW_t^{\,i},\qquad
dW^i\,dW^j=\rho_{ij}\,dt .$$
Parameters: scales $\sigma_i$, decays $\lambda_i$, driver correlations $\rho_{ij}$.

## 4. Instantaneous covariance of increments

With $\tau_a=T_a-t$ and $\mathbb E_t[dW^i dW^j]=\rho_{ij}\,dt$, the instantaneous covariance rate is
$$C(\tau_a,\tau_b)=\sum_{i,j}\rho_{ij}\,\sigma_i\sigma_j\,e^{-\lambda_i\tau_a-\lambda_j\tau_b},$$
a smooth surface controlled entirely by $\theta=\{\sigma_i,\lambda_i,\rho_{ij}\}$ — the object we fit.

## 5. Matrix form $C=APA^\top$

For a grid $\tau_1,\dots,\tau_n$ set $A_{ai}=\sigma_i e^{-\lambda_i\tau_a}$ $(n\times k)$ and
$P=[\rho_{ij}]$. Then
$$\mathbf C = A\,P\,A^\top,$$
symmetric, **PSD whenever $P$ is**, **rank $\le k$**, linear in $P$ and nonlinear only through the
$\lambda_i$.

## 6. Sanity checks

- **Variance term structure** (diagonal): $C(\tau,\tau)=\sum_{ij}\rho_{ij}\sigma_i\sigma_j
  e^{-(\lambda_i+\lambda_j)\tau}$ — a sum of exponentials, good for seeding the $\lambda_i$.
- **One factor is degenerate:** $C=\sigma_1^2 e^{-\lambda_1(\tau_a+\tau_b)}$ is rank-1, so
  $\mathrm{Corr}\equiv 1$. Hence $k\ge 2$; $k=3$ (level / slope / curvature) is the textbook choice.

## 7. From continuous time to the fitted target

Over a step $\Delta t$ the coefficients are ~frozen, so
$\mathrm{Cov}(\Delta f(\tau_a),\Delta f(\tau_b))\approx C(\tau_a,\tau_b)\,\Delta t$. We fit $\theta$ so
$C_\theta\approx\hat\Sigma$ on the **per-day** sample covariance (no division by $\Delta t$), keeping the
same scale as the other estimators.

## 8. Step by step: what `hjm_kernel.py` does

The cells below reproduce the estimator's internals, then cross-check against the production class.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

from src.data_loader import load_rates
from src.tenor_graph import tenor_to_years

INPUTS = ROOT / "inputs"
np.set_printoptions(precision=4, suppress=True)

### 8.1 Load data → sample covariance $\hat\Sigma$ and the $\tau$ grid

The fit target is the **naïve** (sample) covariance of daily par-rate changes; $\tau$ is each tenor's
time-to-maturity in years, read from the column labels via `tenor_to_years`.

In [ ]:
CURVE   = "CAD"
rates   = load_rates(INPUTS / CURVE / "curve_data.csv")
changes = rates.diff().dropna()                  # daily increments Δf
S       = np.cov(changes.values.T)               # naïve target  (n, n)
tau     = np.array([tenor_to_years(c) for c in rates.columns])
n       = len(tau)

print(f"{changes.shape[0]} daily changes x {n} tenors")
print("tenors:", list(rates.columns))
print("tau (yrs):", tau)

### 8.2 Unconstrained reparameterization

The optimizer searches a free vector $\theta$; constraints hold through the mapping: $\sigma_i=e^{s_i}>0$;
$\lambda_i=\sum_{j\le i}\mathrm{softplus}(\ell_j)$ (positive, increasing — kills factor relabeling); a
unit-norm lower-triangular $L$ gives $P=LL^\top$, a valid correlation matrix.

In [ ]:
def softplus(x):
    return np.logaddexp(0.0, x)

def unpack(theta, k):
    sigma = np.exp(theta[:k])                       # σ_i > 0
    lam   = np.cumsum(softplus(theta[k:2*k]))       # λ_i increasing & > 0
    L = np.eye(k)
    L[np.tril_indices(k, -1)] = theta[2*k:]
    L /= np.linalg.norm(L, axis=1, keepdims=True)   # P = L Lᵀ is a correlation matrix
    return sigma, lam, L @ L.T

### 8.3 Model $C=APA^\top$ and the weighted-Frobenius objective

We fit each unique entry once (upper triangle). `weighting="uniform"` matches covariance *levels*;
`weighting="corr"` divides residuals by $\sqrt{\hat\Sigma_{aa}\hat\Sigma_{bb}}$ to fit the correlation
*shape*.

In [ ]:
def model(theta, k):
    sigma, lam, P = unpack(theta, k)
    A = sigma * np.exp(-np.outer(tau, lam))    # A[a,i] = σ_i e^{-λ_i τ_a},  shape (n, k)
    return A @ P @ A.T

iu     = np.triu_indices(n)                    # unique (a, b) entries
target = S[iu]

def resid(theta, k, sw):
    return sw * (model(theta, k)[iu] - target)

### 8.4 Initialization

The structured seed spreads the decays $\lambda_i$ geometrically from slow (whole-curve *level*) to fast
(*short-end*), sets each $\sigma_i$ to a fraction of the average diagonal, and starts uncorrelated
($P=I$). We also support a plain **all-zeros** start ($\sigma_i=1$, evenly spaced $\lambda$, $P=I$); §9.4
compares the two.

In [ ]:
def init(k, jitter, rng, zero=False):
    if zero:
        theta0 = np.zeros(2*k + k*(k-1)//2)
    else:
        s0   = np.full(k, np.log(np.sqrt(max(np.diag(S).mean(), 1e-12) / k)))
        incr = np.maximum(np.diff(np.geomspace(1/tau.max(), 1/tau.min(), k), prepend=0.0), 1e-6)
        l0   = np.log(np.expm1(incr))              # softplus⁻¹ of the per-factor λ increments
        theta0 = np.concatenate([s0, l0, np.zeros(k*(k-1)//2)])
    return theta0 + rng.normal(scale=0.5, size=theta0.shape) if jitter else theta0

### 8.5 Fit by trust-region least squares (with restarts)

`least_squares` (`trf`) minimizes the residual vector; a few seeded random restarts guard against local
minima and we keep the lowest-cost solution.

In [ ]:
def fit_hjm(k, weighting="uniform", n_restarts=4, zero_init=False):
    if weighting == "corr":
        d  = np.sqrt(np.diag(S)); sw = 1.0 / np.sqrt(np.outer(d, d)[iu])
    else:
        sw = np.ones(len(target))
    rng, best = np.random.default_rng(0), None
    for r in range(n_restarts):
        sol = least_squares(resid, init(k, r > 0, rng, zero_init), args=(k, sw), method="trf")
        if best is None or sol.cost < best.cost:
            best = sol
    return best

best = fit_hjm(2); sigma, lam, P = unpack(best.x, 2); C2 = model(best.x, 2)
print("sigma ", sigma, "\nlambda", lam, "\nP\n", P)
print("rel. Frobenius error vs naïve cov: %.4f" % (np.linalg.norm(C2 - S) / np.linalg.norm(S)))

### 8.6 Cross-check against the production estimator

Same algorithm and seed — the manual fit reproduces `HJMKernelEstimator` (default $k=2$).

In [ ]:
from src.estimators import get_estimator

est    = get_estimator("hjm_kernel")
C_prod = est.fit(changes)
print("matches manual fit:", np.allclose(C2, C_prod))
print("fitted params:", {key: np.round(v, 4) for key, v in est.params_.items() if key != "k"})

## 9. Choosing the number of factors $k$

### 9.1 How many factors does the data want?

The eigenvalue spectrum of $\hat\Sigma$ shows how concentrated the variance is — for rates the first 2–3
PCs (level / slope / curvature) explain essentially all of it.

In [ ]:
evals = np.linalg.eigvalsh(S)[::-1]
cum   = np.cumsum(evals) / evals.sum()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar(range(1, n + 1), evals)
ax[0].set(title="Eigenvalues of naïve covariance", xlabel="component", ylabel="variance")
ax[1].plot(range(1, n + 1), cum, "o-"); ax[1].axhline(0.99, ls="--", c="grey")
ax[1].set(title="Cumulative variance explained", xlabel="# factors", ylim=(0.9, 1.005))
for j in (1, 2, 3):
    ax[1].annotate(f"{cum[j-1]:.4f}", (j, cum[j-1]), textcoords="offset points", xytext=(4, -10))
plt.tight_layout(); plt.show()

### 9.2 Fit quality vs $k$

Fit each $k=1,\dots,5$ against the $n(n+1)/2$ free entries of the naïve covariance. The rank caps at $k$,
and **$k=1$ is degenerate**: its minimum off-diagonal correlation is exactly 1.

In [ ]:
def corr_of(M):
    d = np.sqrt(np.diag(M)); return M / np.outer(d, d)

ks, Cs, rows = [1, 2, 3, 4, 5], {}, []
for k in ks:
    Cs[k] = model(fit_hjm(k).x, k)
    rows.append({
        "k": k,
        "params": 2*k + k*(k-1)//2,
        "rel_frob_err": np.linalg.norm(Cs[k] - S) / np.linalg.norm(S),
        "rank": int(np.linalg.matrix_rank(Cs[k], tol=1e-10)),
        "min_offdiag_corr": corr_of(Cs[k])[np.triu_indices(n, 1)].min(),
    })
print("naïve covariance has", n*(n+1)//2, "free entries")
summary = pd.DataFrame(rows).set_index("k")
summary.round(4)

### 9.3 Covariance heatmaps: naïve vs HJM $k=1,\dots,5$

A shared color scale makes the fits directly comparable. The HJM matrices smooth the naïve estimate:
$k=1$ is visibly too flat, and from $k=3$ on the panels are nearly indistinguishable from naïve.

In [ ]:
mats = [("naïve", S)] + [(f"HJM k={k}", Cs[k]) for k in ks]
vmin = min(M.min() for _, M in mats)
vmax = max(M.max() for _, M in mats)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (name, M) in zip(axes.flat, mats):
    im = ax.imshow(M, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(name)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(rates.columns, rotation=90, fontsize=6)
    ax.set_yticklabels(rates.columns, fontsize=6)
fig.colorbar(im, ax=axes, shrink=0.7, label="covariance")
plt.show()

### 9.4 Initialization: structured vs all-zeros

The objective is non-convex, so the seed can matter. We refit each $k$ from the plain all-zeros start and
compare the converged relative error against the structured seed.

In [ ]:
rows = []
for k in ks:
    rows.append({
        "k": k,
        "structured_init": np.linalg.norm(model(fit_hjm(k).x, k) - S) / np.linalg.norm(S),
        "zeros_init":      np.linalg.norm(model(fit_hjm(k, zero_init=True).x, k) - S) / np.linalg.norm(S),
    })
pd.DataFrame(rows).set_index("k").round(4)

### 9.5 Variance term structure: naïve vs fitted

The diagonal $C(\tau,\tau)$ is a sum of exponentials. $k=1$ cannot bend enough; $k=2$ already tracks the
naïve variances closely.

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.plot(tau, np.diag(S), "ko", label="naïve")
for k in (1, 2, 3):
    plt.plot(tau, np.diag(Cs[k]), "-", label=f"HJM k={k}")
plt.xlabel("time to maturity (yrs)"); plt.ylabel("variance C(τ, τ)")
plt.title("Variance term structure"); plt.legend(); plt.grid(alpha=.3); plt.show()

### 9.6 Correlation with a reference tenor

Correlation of the 2Y with every other tenor. $k=1$ is flat at 1 (the §6 degeneracy); $k\ge2$ captures
the decorrelation as tenors separate.

In [ ]:
Cnaive = corr_of(S)
anchor = list(rates.columns).index("2Y")

plt.figure(figsize=(6.5, 4))
plt.plot(tau, Cnaive[anchor], "ko", label="naïve")
for k in (1, 2, 3):
    plt.plot(tau, corr_of(Cs[k])[anchor], "-", label=f"HJM k={k}")
plt.xlabel("time to maturity (yrs)"); plt.ylabel("correlation with 2Y")
plt.title("Correlation of 2Y with other tenors"); plt.legend(); plt.grid(alpha=.3); plt.show()

### 9.7 Fitted factor shapes ($k=3$)

Each column $A_{\cdot i}=\sigma_i e^{-\lambda_i\tau}$ is one factor's loading. The ordering constraint
yields **level** (slow decay), **slope**, and **short-end / curvature** (fast decay) — the classic
decomposition.

In [ ]:
b3 = fit_hjm(3); sig3, lam3, P3 = unpack(b3.x, 3)
A3 = sig3 * np.exp(-np.outer(tau, lam3))

plt.figure(figsize=(6.5, 4))
for i in range(3):
    plt.plot(tau, A3[:, i], "o-", label=f"factor {i+1}:  σ={sig3[i]:.3f}, λ={lam3[i]:.3f}")
plt.xlabel("time to maturity (yrs)"); plt.ylabel("loading  σ_i · e^(−λ_i τ)")
plt.title("Fitted factor loading shapes (k=3)"); plt.legend(); plt.grid(alpha=.3); plt.show()
print("factor correlations P:\n", P3.round(3))

### 9.8 Weighting: levels vs correlation shape

`uniform` minimizes error on covariance *levels* (dominated by the highest-variance tenors); `corr`
spreads effort over the correlation surface. Here the per-tenor variances span a narrow range, so the
two fits are nearly identical.

In [ ]:
for w in ("uniform", "corr"):
    C = model(fit_hjm(2, weighting=w).x, 2)
    cov_err  = np.linalg.norm(C - S) / np.linalg.norm(S)
    corr_err = np.linalg.norm(corr_of(C) - Cnaive) / np.linalg.norm(Cnaive)
    print(f"{w:8s}  cov rel.err = {cov_err:.4f}   corr rel.err = {corr_err:.4f}")

## 10. Exploring a Bayesian fit by MCMC

The §8 fit returns a single point estimate. Reading the weighted least-squares objective as a **Gaussian
log-likelihood** (least squares ⟺ Gaussian errors) and adding a weak prior on the unconstrained $\theta$
turns the fit into a **posterior** we can sample — yielding uncertainty on the parameters and on the
fitted covariance, not just a point. We use a transparent random-walk **Metropolis** sampler (no extra
dependencies) that reuses the exact `resid` from §8.

### 10.1 Likelihood, prior, and the sampler

The noise scale $s$ is fixed to the RMS misfit at the LS optimum (the rank-$k$ model cannot reproduce
$\hat\Sigma$ exactly, so $s$ measures that structural misfit). The prior is a broad $N(0,5^2)$ on each
component of $\theta$ — negligible against the sharp likelihood but enough to keep weakly-identified
directions proper. The chain is warm-started at the LS optimum and proposals use the Gauss–Newton
posterior covariance $\Sigma_\text{prop}\propto s^2(J^\top J)^{-1}$ from the LS Jacobian, giving a healthy
~30% acceptance and fast mixing (traces below).

In [ ]:
def log_post(theta, k, sw, s, prior_sd=5.0):
    r = resid(theta, k, sw)
    return -0.5*np.sum((r/s)**2) - 0.5*np.sum((theta/prior_sd)**2)   # Gaussian lik. + weak prior

def run_mcmc(k, n_steps=40000, burn=5000, thin=20, seed=1):
    sw = np.ones(len(target))                         # uniform weighting, as in §8
    ls = fit_hjm(k)                                   # LS optimum: warm start + proposal scale
    s  = np.sqrt(np.mean(resid(ls.x, k, sw)**2))      # RMS misfit -> Gaussian noise scale
    H  = ls.jac.T @ ls.jac + 1e-8*np.eye(ls.x.size)   # Gauss-Newton Hessian (× 1/s²)
    Lp = np.linalg.cholesky((2.4**2/ls.x.size) * s**2 * np.linalg.inv(H))
    rng = np.random.default_rng(seed)
    th, lp = ls.x.copy(), log_post(ls.x, k, sw, s)
    chain, acc = np.empty((n_steps, th.size)), 0
    for t in range(n_steps):
        prop = th + Lp @ rng.standard_normal(th.size)
        lp_p = log_post(prop, k, sw, s)
        if np.log(rng.random()) < lp_p - lp:
            th, lp, acc = prop, lp_p, acc + 1
        chain[t] = th
    return chain[burn::thin], acc/n_steps, ls

K = 2
chain, acc_rate, ls = run_mcmc(K)
post  = np.array([np.r_[unpack(th, K)[0], unpack(th, K)[1], unpack(th, K)[2][0, 1]] for th in chain])
err   = np.array([np.linalg.norm(model(th, K) - S) / np.linalg.norm(S) for th in chain])
names = ["σ1", "σ2", "λ1", "λ2", "ρ12"]
print(f"k={K}: {len(chain)} posterior draws, acceptance {acc_rate:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))
for a, j in zip(ax, (3, 4)):
    a.plot(post[:, j], lw=.6); a.set_title(f"trace: {names[j]}"); a.set_xlabel("draw")
plt.tight_layout(); plt.show()

### 10.2 Posterior parameters vs the point estimate

Mapping each draw back through `unpack` gives posteriors for $\sigma_i,\lambda_i,\rho_{12}$ and the fit
error. They concentrate tightly around the §8 estimate (dashed): the scales $\sigma_i$ are pinned to a
couple of percent, while the decays $\lambda_i$ and the driver correlation $\rho_{12}$ carry the most
relative uncertainty.

In [ ]:
ls_sig, ls_lam, ls_P = unpack(ls.x, K)
ls_vals = np.r_[ls_sig, ls_lam, ls_P[0, 1]]
ls_err  = np.linalg.norm(model(ls.x, K) - S) / np.linalg.norm(S)

cols   = [post[:, j] for j in range(5)] + [err]
labels = names + ["rel. Frobenius error"]
refs   = list(ls_vals) + [ls_err]

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c, lab, r in zip(axes.flat, cols, labels, refs):
    ax.hist(c, bins=40, color="steelblue", alpha=.8)
    ax.axvline(r, color="k", ls="--", lw=1.2); ax.set_title(lab)
fig.suptitle("Posterior distributions (dashed = §8 least-squares estimate)")
plt.tight_layout(); plt.show()

pd.DataFrame({
    "LS":        refs,
    "post_mean": list(post.mean(0)) + [err.mean()],
    "q2.5":      list(np.percentile(post, 2.5, axis=0)) + [np.percentile(err, 2.5)],
    "q97.5":     list(np.percentile(post, 97.5, axis=0)) + [np.percentile(err, 97.5)],
}, index=names + ["rel_frob_err"]).round(4)

### 10.3 Posterior predictive: variance term structure

Propagating the posterior through the model gives a credible band for the diagonal $C(\tau,\tau)$. The
band is narrow — parameter uncertainty is small — so the visible gaps to the naïve points (§9.5) are
**structural misfit** of the rank-$k$ model, not statistical uncertainty in the fit.

In [ ]:
diag_draws    = np.array([np.diag(model(th, K)) for th in chain])
lo, mid, hi   = np.percentile(diag_draws, [2.5, 50, 97.5], axis=0)

plt.figure(figsize=(6.5, 4))
plt.fill_between(tau, lo, hi, alpha=.25, color="steelblue", label="95% credible band")
plt.plot(tau, mid, "-", color="steelblue", label="posterior median")
plt.plot(tau, np.diag(S), "ko", label="naïve")
plt.xlabel("time to maturity (yrs)"); plt.ylabel("variance C(τ, τ)")
plt.title(f"Posterior-predictive variance term structure (k={K})")
plt.legend(); plt.grid(alpha=.3); plt.show()